# Vinograd, Nair et al. (2024): stored rSLDS weights and holographic photostimulation

Paper: Vinograd A, Nair A, Kim JH, Linderman SW, Anderson DJ (2024). Causal evidence of a line
attractor encoding an affective state. Nature 634, 910-918.
[doi:10.1038/s41586-024-07915-x](https://doi.org/10.1038/s41586-024-07915-x). The claims file was
written from the version of record (open access, corrected 2025) with its Methods and the legends of
the Extended Data figures.

Dataset: [DANDI:001037](https://dandiarchive.org/dandiset/001037/0.260609.1402), version
0.260609.1402. The paper cites the dandiset without a version. Of the three published versions,
0.240816.1841 holds 2 files, 0.260529.1406 holds 9 (no miniscope mice except M30L) and 0.260609.1402
holds 13: the five miniscope mice and the seven two-photon mice. It is the most recent and the only
one that holds the data of every claim the dandiset can test, so it is pinned. The data check below
lists the files of each version.

This entry has four notebooks. This one tests the claims that use the rSLDS factor weights stored
in the two-photon files and the photostimulation sessions; `rslds_fits.ipynb` fits rSLDS models;
`cross_validation.ipynb` cross-validates them; `network_model.ipynb` simulates the spiking network
of Figure 4.

| Claim | Where | Status |
|---|---|---|
| methods-subspace-angle, fig1m-achw-x1-longer, fig2d-x1-integration, fig2gh-x1-moves-along-attractor, ed7f-activated-x1-natural-levels, ed8a-x1-integration-8s, ed8gi-x1-stim-no-x2, fig2k-x2-no-integration, ed8b-x2-no-integration-8s, ed8jl-x2-stim-little-x1, fig2no-x2-off-manifold, ed9d-attractor-finite | this notebook | tested |
| ed1n-engaging-x1-slower, ed1p-observing-x1-slower, ed3c-x1-activity-observe-vs-engage, ed3e-achw-full-session, ed3f-achw-after-removal, ed3g-decode-observed-attack, ed3h-decode-engaged-from-observed, ed4c-x1-neuron-overlap, ed4d-weight-dot-product, ed4g-x2-more-spontaneous, ed4hi-amplitude-tau-similar, fig1e-2p-x1-slower, fig1f-line-attractor-score, x1-time-constant-range, ed8c-model-integrates | rslds_fits.ipynb | tested |
| methods-cvr2 | cross_validation.ipynb | run, then blocked: methods underspecified |
| fig4ce-excitatory-network | network_model.ipynb | run, then blocked: methods underspecified (simulation, no recorded data) |
| ed1m-x1-observe-attack-vs-sniff | | blocked: data missing (sniffing of the SW mouse is annotated in 1 of 5 mice) |
| ed5e-motion-explains-little, ed5hj-no-behaviour-change | | blocked: data missing (no face video or motion SVDs) |
| ed6g-offtarget-15um | | blocked: data missing (no calibration recordings) |
| ed6k-no-spatial-clustering, ed7bc-no-spatial-decoding | | blocked: data missing (no cell positions) |
| ed8ef-x1-scales-with-power | | blocked: data missing (no 1 mW sessions) |
| ed9i-points-along-line-attractive | | blocked: data missing (no x1-then-x2 session) |
| ed9mn-random-neurons-no-effect | | blocked: data missing (no random-neuron sessions) |
| ed10c-followers-integrate, ed10g-follower-decoding | | blocked: data missing (no cell positions for the 50 um exclusion) |
| ed10hi, ed10lm, ed10p, ed10r, fig3d, fig3e, fig3i, fig3j, fig4a, fig5g, ed12, fig5f, ed12eh | | blocked: data missing (no unitary stimulation sessions) |
| fig4hj-inhibition-slow-dense | | blocked: data missing (depends on fig4a-dense-x1-connectivity) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import requests

import linattr as L

warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
D = {s: L.load_2p(s) for s in L.TWO_PHOTON}
print(f"dandiset {L.DANDISET} version {L.VERSION}: loaded {len(D)} two-photon mice in {time.time() - started:.0f} s")

## Data check

In [ ]:
print("Files per published version:")
for v in ["0.240816.1841", "0.260529.1406", L.VERSION]:
    a = L.assets(v)
    print(f"  {v}: {len(a)} files")
    for p in sorted(a):
        print(f"      {p}  asset {a[p][:8]}")
print("\nTwo-photon mice (paper: 9 mice in Fig. 1, 8 with x1 and 7 with x2 group activation):")
for s, d in D.items():
    i1, i2 = L.x1_x2_factor(d)
    print(f"  {s}: {d['W'].shape[0]} cells; x1 is stored factor {i1 + 1}; BALB/c entry marker {d['entry_s']} s; "
          f"behaviour intervals: {sorted(d['beh']) or 'none'}")
    for ses, Y in d["sessions"].items():
        on = d["stim"].get(ses)
        print(f"      {ses:38s} {Y.shape[0]:5d} frames x {Y.shape[1]} cells, mean {Y.mean():6.3f}, sd {Y.std():5.3f}, "
              f"range {Y.min():6.2f} to {Y.max():6.2f}; onsets (frames) {list(on) if on is not None else '-'}; "
              f"targeted {[i + 1 for i in d['activated'].get(ses, [])] or '-'}")
f = L.open_nwb(L.asset_path("M70", "2p"))
print("\nSampling rate of every TimeSeries (Hz):", sorted({float(f[f"processing/ophys/{k}/starting_time"].attrs["rate"]) for k in f["processing/ophys"]}))
print("Projection baselines (1-based inclusive frames) and smoothing windows:",
      sorted({(m["b0"], m["b1"], m["smooth"]) for d in D.values() for m in d["proj"].values()}))
n_x1 = sum("activation_integration_neurons" in d["sessions"] for d in D.values())
n_x2 = sum("activation_novel_neurons" in d["sessions"] for d in D.values())
n_x1_8 = sum("activation_integration_neurons_8isi" in d["sessions"] for d in D.values())
n_x2_8 = sum("activation_novel_neurons_8isi" in d["sessions"] for d in D.values())
print(f"Mice per session: x1 20 s ISI {n_x1}, x2 20 s ISI {n_x2}, x1 8 s ISI {n_x1_8}, x2 8 s ISI {n_x2_8}")

The two-photon files hold 7 mice, not the paper's 9 (Fig. 1) or 8 (x1 group activation): there are
7 x1 and 7 x2 activation sessions at 20 s ISI, 6 x1 and 4 x2 at 8 s ISI. The z-scored activity has
mean near 0 and SD near 1, as its description says. Only M70 has behaviour intervals for the
observation session; the others have a single BALB/c entry marker. Next: the earlier M70 file
(`sub-M70_ses-None_behavior.nwb`), and whether the targeted neurons are the ones the stored weights
rank highest.

In [ ]:
fl = L.open_nwb(L.asset_path("M70", "legacy"))
Yl = fl["processing/ophys/NeuralTrace/data"][:]
Wl = np.c_[fl["analysis/rSLDS/latent factor 1"][:], fl["analysis/rSLDS/latent factor 2"][:]]
Y = D["M70"]["sessions"]["observation"]
r = [np.corrcoef(Yl[:len(Y), j], Y[:, j])[0, 1] for j in range(Y.shape[1])]
print(f"Earlier M70 file: {Yl.shape[0]} frames x {Yl.shape[1]} cells; current M70 observation: {Y.shape[0]} x {Y.shape[1]}")
print(f"Per-cell correlation over the first {len(Y)} frames: median {np.median(r):.3f}, min {np.min(r):.3f}")
print("BALB/c intervals, earlier file:", L.read_intervals(fl["processing/behavior/BehavioralEpochs/Ch1: Balc in cage"]))
print("BALB/c intervals, current file:", D["M70"]["beh"]["balbc_in_cage"])
print("Correlation of the earlier file's two factors with the current file's (rows earlier, columns current):")
print(np.round(np.corrcoef(np.c_[Wl, D["M70"]["W"]].T)[:2, 2:], 3))
print("\nRank (by absolute stored weight on its own dimension, 1 = largest) of each targeted neuron:")
for s, d in D.items():
    i1, i2 = L.x1_x2_factor(d)
    row = []
    for ses in ["activation_integration_neurons", "activation_novel_neurons"]:
        w = np.abs(d["W"][:, i1 if "integration" in ses else i2])
        rank = np.argsort(np.argsort(-w)) + 1
        row.append(f"{'x1' if 'integration' in ses else 'x2'} targets {sorted(rank[d['activated'][ses]].tolist())}")
    print(f"  {s}: " + "; ".join(row) + f" (of {len(w)} cells)")
for c in range(2):
    rank = np.argsort(np.argsort(-np.abs(Wl[:, c]))) + 1
    print(f"  M70 with the earlier file's factor {c + 1}: x1 targets {sorted(rank[D['M70']['activated']['activation_integration_neurons']].tolist())}")

The earlier M70 file is the same recording as the current M70 observation session (per-cell
correlations printed above), so M70 is counted once. The current file stops the session at 300 s,
during the second intruder visit, which the earlier file follows to 350 s. The two files' factor
weights are different fits. In six mice the targeted x1 neurons are among the highest x1 weights;
in M70 they rank 33rd to 55th of 65 by the stored x1 weight, so the stored M70 weights are not the
ones the targets were chosen from (Methods: "Neurons were selected for targeted photostimulation
based on ... their weights from the rSLDS model").

Clock check. The stimulation onsets are stored as frame numbers. If they share the activity's clock,
the targeted neurons rise right after each stored onset and not after onsets shifted in time.

In [ ]:
def evoked(d, ses, shift):
    Y = d["sessions"][ses][:, d["activated"][ses]].mean(1)
    on = d["stim"][ses] + shift
    return np.mean([Y[o:o + L.PULSE_FRAMES].mean() - Y[o - 20:o].mean() for o in on if o - 20 >= 0 and o + 20 <= len(Y)])

for ses in ["activation_integration_neurons", "activation_novel_neurons"]:
    for shift in [0, -60, 60]:
        v = [evoked(d, ses, shift) for d in D.values()]
        print(f"{ses:32s} onsets shifted {shift:+4d} frames: targeted-cell response (2 s after minus 2 s before) "
              f"mean {np.mean(v):5.2f} z, per mouse {np.round(v, 2).tolist()}")
for shift in [0, -300, 300]:
    v = []
    for d in D.values():
        x1 = L.projection(d, "observation", "x1")
        e = int(round(d["entry_s"] * L.RATE)) + shift
        if 100 <= e and e + 300 <= len(x1):
            v.append(x1[e:e + 300].mean() - x1[e - 100:e].mean())
    print(f"Observation x1 projection, 30 s after minus 10 s before the BALB/c entry marker shifted {shift:+d} frames: "
          f"mean {np.mean(v):5.2f} over {len(v)} mice")

Targeted cells respond within 2 s of the stored onsets (0.50 z for x1 targets, 0.37 z for x2
targets) and not at onsets shifted by 6 s (-0.16 to 0.00 z), and the x1 projection rises after the
stored BALB/c entry marker (2.65) more than after markers shifted by 30 s (0.41 and -0.09), so the
stimulation frames and the entry marker are consistent with the activity's clock. One exception:
in M70 the five neurons listed as targeted in the x1 activation session do not respond at the
onsets (-0.01 z), which with their low x1 weights suggests that M70's `ActivatedNeurons` indices, or
its weights, do not refer to the cells of the activity matrix. M70 is kept, as the projections do
not use the target list, and this is recorded as a finding.

## methods-subspace-angle

> The x1 and x2 dimensions are nearly orthogonal, with a subspace angle of 88.1 +/- 0.87 degrees
> across 9 mice.

In [ ]:
ang = {s: L.subspace_angle(d["W"][:, 0], d["W"][:, 1]) for s, d in D.items()}
for s, a in ang.items():
    print(f"{s}: {a:.2f} deg")
v = np.array(list(ang.values()))
print(f"mean {v.mean():.2f} +/- {v.std(ddof=1) / np.sqrt(len(v)):.2f} (s.e.m.), sd {v.std(ddof=1):.2f}, n = {len(v)} mice; "
      f"expected 88.1 +/- 0.87, so {88.1 - 0.87:.2f} to {88.1 + 0.87:.2f}")
print(f"earlier M70 file: {L.subspace_angle(Wl[:, 0], Wl[:, 1]):.2f} deg")

**Result: reproduced.** The angle between the stored x1 and x2 weight vectors is 86.4 to 89.9 deg in
the 7 mice, mean 88.55 (s.e.m. 0.49), inside 87.23 to 88.97. Two mice of the paper's nine are not in the dandiset.

## fig1m-achw-x1-longer

> Neurons contributing to x1 have longer autocorrelation half-widths than those contributing to x2 in
> head-fixed observation.

Five neurons per dimension and mouse, those with the largest absolute stored weight; the half-width
is the first lag at which the autocorrelation (Methods, eqs 7 and 8) falls to 0.5 or below.

In [ ]:
a1, a2 = [], []
for s, d in D.items():
    i1, i2 = L.x1_x2_factor(d)
    Y = d["sessions"]["observation"]
    h1 = [L.achw(Y[:, j]) for j in L.top_neurons(d["W"][:, i1])]
    h2 = [L.achw(Y[:, j]) for j in L.top_neurons(d["W"][:, i2])]
    a1.append(np.nanmean(h1)); a2.append(np.nanmean(h2))
    print(f"{s}: x1 {np.round(h1, 1).tolist()} s, x2 {np.round(h2, 1).tolist()} s")
a1, a2 = np.array(a1), np.array(a2)
print(f"\nPer-mouse means: x1 {np.round(a1, 1).tolist()}, x2 {np.round(a2, 1).tolist()}")
print(f"x1 longer in {(a1 > a2).sum()} of {len(a1)} mice; mean x1 {a1.mean():.1f} s, x2 {a2.mean():.1f} s; "
      f"median difference {np.median(a1 - a2):.1f} s")
print(f"unpaired Mann-Whitney p = {L.mwu(a1, a2):.4f}; paired signed-rank p = {L.wsr(a1, a2):.4f}")
for n in [3, 10]:
    b1 = [np.nanmean([L.achw(d['sessions']['observation'][:, j]) for j in L.top_neurons(d['W'][:, L.x1_x2_factor(d)[0]], n)]) for d in D.values()]
    b2 = [np.nanmean([L.achw(d['sessions']['observation'][:, j]) for j in L.top_neurons(d['W'][:, L.x1_x2_factor(d)[1]], n)]) for d in D.values()]
    print(f"top {n} neurons per dimension: x1 longer in {sum(np.array(b1) > np.array(b2))} of 7; "
          f"Mann-Whitney p = {L.mwu(b1, b2):.4f}, signed-rank p = {L.wsr(b1, b2):.4f}")

**Result: reproduced, under the paired reading.** x1 neurons have longer half-widths than x2
neurons in all 7 mice (means 20.8 against 13.7 s); the paired signed-rank test gives p = 0.016, the
smallest value possible with 7 mice, while the unpaired Mann-Whitney test gives p = 0.16. The paired
test is adopted as an assumption: the two values come from the same mouse and the Methods use
Mann-Whitney "for binary paired samples"; the printed 0.0078 fits either test at n = 9.

## fig2d-x1-integration

> Photostimulation of five x1 neurons with four pulses at 20 s ISI produces integration in the x1
> dimension, with ISI activity rising with each pulse and not returning to baseline.

Projection as in the sample code. Readings (claims header): windows Wa (end of pulse to next onset)
and Wb (onset to onset); Dunn comparisons Ca (each ISI against baseline) and Cb (each against the
one before). Kruskal-Wallis over the five periods with one value per mouse, then Dunn's test with a
Bonferroni-type adjustment over the four comparisons.

In [ ]:
def isi_test(session, dim, label, mice=None):
    rows = {}
    for R in ["Wa", "Wb"]:
        M, names = [], []
        for s, d in D.items():
            if session not in d["sessions"] or (mice and s not in mice):
                continue
            M.append(L.periods(L.projection(d, session, dim), d["stim"][session], R))
            names.append(s)
        M = np.array(M)
        for C in ["Ca", "Cb"]:
            H, pkw, ps = L.dunn(list(M.T), L.PAIRS[C])
            rows[(R, C)] = (M, ps)
            print(f"{label} {R} {C}: n = {len(M)}; means baseline, ISI1-4 = {np.round(M.mean(0), 2).tolist()}; "
                  f"Kruskal-Wallis H = {H:.2f}, p = {pkw:.4f}; Dunn p = {[round(p, 4) for p in ps]}; "
                  f"significant {sum(p < 0.05 for p in ps)} of 4")
        print(f"   per mouse ({R}): " + "; ".join(f"{n} {np.round(m, 2).tolist()}" for n, m in zip(names, M)))
    return rows

fig2d = isi_test("activation_integration_neurons", "x1", "x1 projection, x1 activation 20 s ISI")
for k, (M, ps) in fig2d.items():
    print(k, "rise ISI1 -> ISI3:", round(M.mean(0)[1], 2), "->", round(M.mean(0)[3], 2),
          "| mice whose ISI4 is above baseline:", int((M[:, 4] > M[:, 0]).sum()), "of", len(M))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
tr = np.array([L.projection(d, "activation_integration_neurons", "x1")[:1190] for d in D.values()])
t = np.arange(tr.shape[1]) / L.RATE
ax.plot(t, tr.mean(0), color="#1b7837")
ax.fill_between(t, tr.mean(0) - tr.std(0, ddof=1) / np.sqrt(len(tr)), tr.mean(0) + tr.std(0, ddof=1) / np.sqrt(len(tr)), color="#1b7837", alpha=0.3)
for o in D["M70"]["stim"]["activation_integration_neurons"]:
    ax.axvline(o / L.RATE, color="#c51b7d", lw=0.8)
ax.set_xlabel("time (s)"); ax.set_ylabel("x1 projection (z)"); ax.set_title("x1 activation, mean +/- s.e.m. over 7 mice")
plt.show()
print(f"plotted range: mean trace {tr.mean(0).min():.2f} to {tr.mean(0).max():.2f}; all points inside the axes")

**Result: reproduced.** Under the readings Wa-Ca and Wb-Ca, three of four Dunn comparisons against
baseline are significant (Wa: p = 0.38, 0.0045, 0.0001, 0.0013) and the mean x1 projection rises from
1.82 in ISI1 to 3.22 in ISI3 (baseline 0.18); ISI4 stays above baseline in every mouse. Under Cb
(each ISI against the one before) only the first comparison could be tested as significant and none
is.

## fig2gh-x1-moves-along-attractor

> x1 photostimulation moves the population activity vector progressively along the line attractor;
> the state-space distance from the initial point grows with each pulse, both at the end of
> stimulation and at the end of each ISI.

State = (x1, x2) projections. t_stim-end = onset + 2 s; t_post-stim = the frame before the next
onset (last pulse: onset + 21 s). Readings: t_initial the mean baseline state or the last baseline
frame; comparisons (a) each pulse against the mean distance of the baseline frames from t_initial,
(b) each pulse against the first (and the first against baseline).

In [ ]:
def distances(d, ses, tinit, which):
    s = np.c_[L.projection(d, ses, "x1"), L.projection(d, ses, "x2")]
    on = d["stim"][ses]; isi = on[1] - on[0]
    p0 = s[:on[0]].mean(0) if tinit == "mean" else s[on[0] - 1]
    out = [np.linalg.norm(s[:on[0]] - p0, axis=1).mean()]
    for i, o in enumerate(on):
        tt = o + L.PULSE_FRAMES if which == "stim" else (on[i + 1] - 1 if i + 1 < len(on) else o + isi + 1)
        out.append(np.linalg.norm(s[min(tt, len(s) - 1)] - p0))
    return np.array(out)

PAIRS_DIST = {"a": [(0, 1), (0, 2), (0, 3), (0, 4)], "b": [(0, 1), (1, 2), (1, 3), (1, 4)]}

def distance_test(ses, label):
    res = {}
    for tinit in ["mean", "last"]:
        for which in ["stim", "post"]:
            M = np.array([distances(d, ses, tinit, which) for d in D.values() if ses in d["sessions"]])
            for C, pairs in PAIRS_DIST.items():
                H, pkw, ps = L.dunn(list(M.T), pairs)
                res[(tinit, which, C)] = (M.mean(0), ps)
                print(f"{label} t_initial={tinit:4s} {which:4s} comparison {C}: means baseline, pulses 1-4 = "
                      f"{np.round(M.mean(0), 2).tolist()}; Dunn p = {[round(p, 4) for p in ps]}; significant {sum(p < 0.05 for p in ps)}")
    return res

fig2gh = distance_test("activation_integration_neurons", "x1 activation")
print()
for tinit in ["mean", "last"]:
    for C in ["a", "b"]:
        ok = []
        for which in ["stim", "post"]:
            m, ps = fig2gh[(tinit, which, C)]
            ok.append((m[3] > m[1], sum(p < 0.05 for p in ps)))
        print(f"reading t_initial={tinit}, comparison {C}: (rise pulse1->3, significant of 4) stim-end {ok[0]}, post-stim {ok[1]}")

**Result: not reproduced.** With t_initial the mean baseline state, the mean distance rises from
pulse 1 to pulse 3 for both metrics (stimulation end 3.01 to 5.20, ISI end 2.15 to 2.80) and three of
four ISI-end comparisons against baseline are significant (p = 0.018, 0.0083, 0.036), but at the end
of stimulation only two of four are (p = 0.0028 and 0.0001). With the last baseline frame as
t_initial and comparison b, three stimulation-end comparisons are significant, but the ISI-end
distances do not rise (2.37 to 2.05) and none is significant. No reading meets the criterion for both
metrics.

## ed7f-activated-x1-natural-levels

> Photoactivated x1 neurons reach peak activity comparable to their peak during observation of
> aggression.

In [ ]:
po, pa = [], []
for s, d in D.items():
    idx = d["activated"]["activation_integration_neurons"]
    po.append(d["sessions"]["observation"][:, idx].max(0).mean())
    pa.append(d["sessions"]["activation_integration_neurons"][:, idx].max(0).mean())
    print(f"{s}: mean peak z of the 5 targeted x1 neurons, observation {po[-1]:.2f}, activation {pa[-1]:.2f}")
print(f"mean observation {np.mean(po):.2f}, activation {np.mean(pa):.2f}, difference {np.mean(pa) - np.mean(po):+.2f} z")
print(f"two-tailed Mann-Whitney p = {L.mwu(po, pa):.4f}  (paired signed-rank, shown as a variant: p = {L.wsr(po, pa):.4f})")

**Result: reproduced.** The targeted x1 neurons peak at 1.96 z during stimulation and 1.65 z during
observation (difference +0.31 z); the Mann-Whitney test finds no difference (p = 0.097), as the
paper's test does (p > 0.05). Seven mice of the paper's eight.

## ed8a-x1-integration-8s

> x1 activation with an 8 s ISI also produces integration in the x1 dimension.

In [ ]:
ed8a = isi_test("activation_integration_neurons_8isi", "x1", "x1 projection, x1 activation 8 s ISI")

**Result: reproduced.** In the 6 mice with an 8 s ISI x1 session, three of four comparisons against
baseline are significant under Wa (p = 0.35, 0.0074, 0.0016, 0.0023) and Wb, and the mean rises from
1.86 in ISI1 to 3.23 in ISI3. The paper has 7 mice.

## ed8gi-x1-stim-no-x2

> x1 stimulation does not evoke appreciable activity in the x2 dimension.

Judged under the reading used for fig2d-x1-integration (Wa or Wb with Ca; both shown).

In [ ]:
ed8gi = isi_test("activation_integration_neurons", "x2", "x2 projection, x1 activation 20 s ISI")

**Result: not reproduced.** During x1 stimulation the x2 projection is above baseline in the third
ISI in 6 of 7 mice (means 0.02, 0.70, 1.04, 0.42 over the ISIs against -0.09 at baseline, Wa), and
the third ISI differs from baseline (Dunn p = 0.036 under Wa-Ca and Wb-Ca, the readings under which
fig2d is reproduced); the paper's test finds no difference. M18R goes negative in every ISI. The x2
response is about a third of the x1 response (fig2d).

## fig2k-x2-no-integration

> Photostimulation of five x2 neurons does not produce integration; x2 activity is transient and
> decays to baseline during each ISI.

In [ ]:
fig2k = isi_test("activation_novel_neurons", "x2", "x2 projection, x2 activation 20 s ISI")

**Result: reproduced.** No comparison differs from baseline or from the previous ISI under any
reading (Kruskal-Wallis p = 0.75 under Wa; all Dunn p = 1.0); mean ISI values -0.47 to 0.06 against
0.01 at baseline.

## ed8b-x2-no-integration-8s

> With an 8 s ISI, x2 activation also produces no integration in x2.

In [ ]:
ed8b = isi_test("activation_novel_neurons_8isi", "x2", "x2 projection, x2 activation 8 s ISI")

**Result: reproduced, under the reading Wa.** In the 4 mice with an 8 s ISI x2 session (the paper
has 7), no comparison is significant under Wa (smallest Dunn p = 0.079), but under Wb the fourth
ISI differs from baseline (p = 0.041). Wa is adopted as an assumption: it is the inter-stimulus
interval in the plain sense, the time between the end of one pulse and the next. The mean x2 level
stays above baseline in every ISI (about 1.6 against 0.1), which the claim's "decays to baseline"
does not describe; with 4 mice the test has little power.

## ed8jl-x2-stim-little-x1

> x2 activation has little effect on x1 activity; in the paper's test two of four comparisons are
> not significant (P = 0.276) and two are (P = 0.0072 and 0.03).

In [ ]:
ed8jl = isi_test("activation_novel_neurons", "x1", "x1 projection, x2 activation 20 s ISI")
for R in ["Wa", "Wb"]:
    m_x2 = ed8jl[(R, "Ca")][0].mean(0)[1:]
    m_x1 = fig2d[(R, "Ca")][0].mean(0)[1:]
    print(f"{R}: x1 mean per ISI under x2 activation {np.round(m_x2, 2).tolist()} against under x1 activation "
          f"{np.round(m_x1, 2).tolist()}; smaller in every ISI: {bool((m_x2 < m_x1).all())}")

**Result: reproduced.** Under x2 activation the x1 projection rises a little (ISI means 0.38 to 0.89
against 0.13 at baseline, Wa), no comparison is significant (smallest p = 0.20), and it is below
the response to x1 activation in every ISI (0.38 to 0.89 against 1.82 to 3.22). The paper finds two
of four significant; the criterion allows up to two.

## fig2no-x2-off-manifold

> x2 activation moves the population vector transiently off the attractor during each pulse, after
> which it relaxes back near its initial location.

In [ ]:
fig2no = distance_test("activation_novel_neurons", "x2 activation")
print()
for tinit in ["mean", "last"]:
    for C in ["a", "b"]:
        ms, ps_s = fig2no[(tinit, "stim", C)]
        mp, ps_p = fig2no[(tinit, "post", C)]
        print(f"t_initial={tinit}, comparison {C}: stimulation-end significant {sum(p < 0.05 for p in ps_s)} of 4, "
              f"ISI-end significant {sum(p < 0.05 for p in ps_p)} of 4, mean ISI-end distance {mp[1:].mean():.2f} "
              f"against stimulation-end {ms[1:].mean():.2f}")

**Result: not reproduced.** At the end of the x2 pulses at most two of four stimulation-end
comparisons are significant under any reading (best: t_initial the mean baseline state, comparison
a, p = 0.036 and 0.0025 for pulses 2 and 3); the criterion needs three of four. With that t_initial
the ISI-end distances are smaller than the stimulation-end distances (mean 1.86 against 2.28) and no
ISI-end comparison is significant, which is the relaxation the claim describes; with the last
baseline frame as t_initial they are larger (2.56 against 1.80).

## ed9d-attractor-finite

> Integration of pulses saturates after the third pulse; a model with a finite attractor fits the x1
> response better than an infinite one.

Reading (a): a step model (each pulse adds the same step to the x1 level from its onset) with the
cumulative level capped at a fitted maximum, against the same model uncapped, fit by least squares to
each mouse's x1 trace from the first onset; mean square error over the whole trace and after the
third pulse; Mann-Whitney over mice. Reading (b): the mean ISI levels, saturation being ISI4 not
above ISI3 while ISI3 is above ISI2.

In [ ]:
from scipy.optimize import minimize_scalar

def step_fit(tr, on, cap):
    start = on[0]
    y = tr[start:]
    k = np.zeros(len(y))
    for o in on:
        k[o - start:] += 1
    def mse_for(c):
        lev = k if c is None else np.minimum(k, c)
        s = (lev @ y) / (lev @ lev)
        return np.mean((y - s * lev) ** 2), s * lev
    if not cap:
        e, pred = mse_for(None)
    else:
        r = minimize_scalar(lambda c: mse_for(c)[0], bounds=(0.5, 4.0), method="bounded")
        e, pred = mse_for(r.x)
    after = on[2] + L.PULSE_FRAMES - start
    return e, np.mean((y[after:] - pred[after:]) ** 2)

fin, inf_ = [], []
for s, d in D.items():
    tr = L.projection(d, "activation_integration_neurons", "x1")
    on = d["stim"]["activation_integration_neurons"]
    fin.append(step_fit(tr, on, True)); inf_.append(step_fit(tr, on, False))
    print(f"{s}: MSE finite {fin[-1][0]:.3f} / after pulse 3 {fin[-1][1]:.3f}; infinite {inf_[-1][0]:.3f} / {inf_[-1][1]:.3f}")
fin, inf_ = np.array(fin), np.array(inf_)
print(f"(a) whole trace: finite lower in {(fin[:, 0] < inf_[:, 0]).sum()} of 7, mean {fin[:, 0].mean():.3f} against "
      f"{inf_[:, 0].mean():.3f}, Mann-Whitney p = {L.mwu(fin[:, 0], inf_[:, 0]):.4f}")
print(f"(a) after the third pulse: finite lower in {(fin[:, 1] < inf_[:, 1]).sum()} of 7, mean {fin[:, 1].mean():.3f} "
      f"against {inf_[:, 1].mean():.3f}, Mann-Whitney p = {L.mwu(fin[:, 1], inf_[:, 1]):.4f}")
for R in ["Wa", "Wb"]:
    M = fig2d[(R, "Ca")][0]
    m = M.mean(0)
    sat = (M[:, 4] <= M[:, 3]) & (M[:, 3] > M[:, 2])
    print(f"(b) {R}: mean ISI2 {m[2]:.2f}, ISI3 {m[3]:.2f}, ISI4 {m[4]:.2f}; pattern holds for the mean: "
          f"{bool(m[4] <= m[3] and m[3] > m[2])}; in {sat.sum()} of {len(M)} mice")

**Result: partially reproduced.** Under reading (a) the capped (finite) model has the lower error in
all 7 mice, significantly over the whole trace (mean 0.95 against 1.81, Mann-Whitney p = 0.038) but
not after the third pulse (1.14 against 1.56, p = 0.26); the paper reports p < 0.001 for both. Under
reading (b) the mean x1 level saturates (ISI2 2.62, ISI3 3.22, ISI4 2.91, Wa), but the pattern holds
in only 2 of 7 mice. Saturation is present in the mean and in the whole-trace fit, but not in every
test the criterion names.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Notebook | Outcome |
|---|---|---|
| methods-subspace-angle | this | reproduced |
| fig1m-achw-x1-longer | this | reproduced (assumed: paired test) |
| fig2d-x1-integration | this | reproduced |
| fig2gh-x1-moves-along-attractor | this | not reproduced |
| ed7f-activated-x1-natural-levels | this | reproduced |
| ed8a-x1-integration-8s | this | reproduced |
| ed8gi-x1-stim-no-x2 | this | not reproduced |
| fig2k-x2-no-integration | this | reproduced |
| ed8b-x2-no-integration-8s | this | reproduced (assumed: window Wa) |
| ed8jl-x2-stim-little-x1 | this | reproduced |
| fig2no-x2-off-manifold | this | not reproduced |
| ed9d-attractor-finite | this | partially reproduced |
| claims of rslds_fits, cross_validation and network_model | those notebooks | see their summaries |
| ed1m, ed5e, ed5hj, ed6g, ed6k, ed7bc, ed8ef, ed9i, ed9mn, ed10c, ed10g, ed10hi, ed10lm, ed10p, ed10r, fig3d, fig3e, fig3i, fig3j, fig4a, fig4hj, fig5g, ed12, fig5f, ed12eh | | blocked: data missing |

How the analysis differs from the paper's: 7 two-photon mice where the paper has 9 (8 for x1
activation, 7 for 8 s ISI); projections, baselines and stimulation frames as the dandiset stores them
and the sample code computes them; ISI windows and Dunn comparisons run under the readings the claims
file names, Dunn's adjustment taken as Bonferroni over the four comparisons.

What was found about the dataset: the dandiset holds the group photostimulation sessions and the
observation sessions with stored rSLDS weights for 7 two-photon mice, but not the unitary
stimulation, 1 mW, random-neuron or x1-then-x2 sessions, cell positions, face video, or calibration
recordings; an earlier M70 file duplicates the current M70 observation session and runs 143 s longer;
in M70 the stored targeted-neuron indices and x1 weights do not fit the stimulation responses.